Solution to [Day 5 Fresh Ingredients](https://adventofcode.com/2025/day/5).

Part1 1 looks for how many in a given list of integers are within a list of ranges. My ranges are sorted by starting numbers, and I do a preliminary check if an integer is less than the minimum start or greater than the maximum stop of all ranges. But apart from this, it's a brute force approach using function *is_in_ranges*.

Part 2. The challenge counting all the numbers in the ranges, taking care not to duplicate numbers and not to fill in gaps. This requires combining overlapping ranges first before finding the number of integers represented = total of last number minus first number in every range. The function *helper_overlaps_in_list* uses *helper_combine_overlaps_1_by_1* to accomplish this. 

-Annette


In [92]:
#Part 1 Function

def is_in_ranges(num: int, ranges: list[range], smallest: int, largest: int)-> bool:
    '''num: a number from a sorted list
    ranges: sorted list of number ranges'''
    if num < smallest or num > largest:
        return False
    for r in ranges:
        if num in r:
            return True
    return False

In [93]:
#Part 1 Solution

ranges = []
nums_to_check = []

#Populate ranges and nums_to_check from file
with open('inputs/day5_input.txt','r') as f:
    lines = f.read().splitlines()

for l in lines:
    try:
        start, end = l.split('-')
        ranges.append(range(int(start),int(end)+1))
    except ValueError:
        try:
            nums_to_check.append(int(l))
        except ValueError:
            continue

#sorted lists
ranges.sort(key=lambda x: x.start) #sort by starting value
nums_to_check.sort() #sorted by value

#minimum and maximum value in all ranges
smallest = min([x.start for x in ranges])
largest = max([x.stop - 1 for x in ranges])  # stop is exclusive, so subtract 1

print(f"Part 1 Answer: {sum([is_in_ranges(n, ranges, smallest, largest) for n in nums_to_check])}")


Part 1 Answer: 720


In [94]:
#Part 2 Functions

'''Steps:
- combine overlapping ranges. This function will have to be used a few times until no overlaps could no longer be found.
- new list of ranges, do subtraction of end-start, add totals'''

def helper_overlapping_1_by_1(r1: range, r2: range) -> list[range]:
    '''combine overlapping ranges. if not overlapping, return both'''
    if r2.start in r1:
        return [range(r1.start, max(r1.stop, r2.stop))]
    elif r1.start in r2:
        return [range(r2.start, max(r1.stop, r2.stop))]
    elif r2.stop in r1:
        return [range(min(r1.start, r2.start), r1.stop)]
    elif r1.stop in r2:
        return [range(min(r1.start, r2.start), r2.stop)]
    else: #no overlap
        return [r1,r2]

def helper_overlaps_in_list(ranges:list[range])-> list[range]:
    '''combine overlapping ranges'''
    new_ranges = [ranges[0]]
    # for i in range(1,len(ranges),1):
    for r2 in ranges[1:]:
        for r1 in new_ranges:
            # print(r1, r2)
            replacement = helper_overlapping_1_by_1(r1,r2)
            if len(replacement)==1:
                new_ranges.remove(r1)
                new_ranges += replacement
                break #match found
        if len(replacement)==2: #no match found
            new_ranges.append(r2)
    return new_ranges

def count_all_integers_in_ranges(ranges:list[range]) -> int:
    all_possible = 0
    for r in ranges:
        all_possible += r.stop
        all_possible -= r.start
    return all_possible



In [95]:
#Part 2 Solution

print('number of original ranges:', len(ranges))
range2 = ranges.copy()

#Go through a loop combining overlapping ranges

starting_length = len(range2) + 1
ending_length = starting_length - 1 # placeholder
while ending_length < starting_length:
    starting_length = ending_length
    range2 = helper_overlaps_in_list(range2)
    ending_length = len(range2)
    print ('  combining overlapping ranges, number of ranges remaining:',ending_length)


print(f"Part 2. Total number of integers in all ranges = or number of fresh ingredients = {count_all_integers_in_ranges(range2)}")

number of original ranges: 195
  combining overlapping ranges, number of ranges remaining: 80
  combining overlapping ranges, number of ranges remaining: 80
Part 2. Total number of integers in all ranges = or number of fresh ingredients = 357608232770687
